# LAB-HW-02 — KV260 Power + Target Discovery

**One problem today: power the KV260 correctly and make the development host discover a real JTAG device.**

Do not write neuron RTL, generate a bitstream, or learn AXI today.

**Prerequisites: LAB-HW-00 and LAB-HW-01.**  
**Project Trace:** RMD-012 · T-HW-002/T-HW-011

## 1. What should be on the desk

- development host that passed LAB-HW-00;
- inventoried KV260;
- **12 V / 3 A**, center-positive supply meeting KV260 requirements;
- USB data cable from the development host to **J4**;
- microSD is not required for JTAG target discovery.

AMD DS986 specifies +12 V / 3 A DC input. A connected USB cable is not proof that the board has main power.

## 2. Preflight

Confirm:

- [ ] KV260 is not yet receiving 12 V;
- [ ] USB cable supports data;
- [ ] USB is connected to **J4 FTDI UART/JTAG**;
- [ ] 12 V supply will connect to **J12**;
- [ ] LAB-HW-00 passed;
- [ ] no unknown external voltage is connected to Pmod/GPIO.

## 3. Connection Map

<svg xmlns="http://www.w3.org/2000/svg" width="760" height="330" viewBox="0 0 760 330" role="img" aria-label="KV260 LAB-HW-02 power and JTAG connection map">
  <rect x="30" y="115" width="170" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="115" y="145" text-anchor="middle" font-size="16">development host</text>
  <text x="115" y="170" text-anchor="middle" font-size="13">Vivado 2026.1</text>

  <rect x="295" y="55" width="180" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="385" y="85" text-anchor="middle" font-size="16">J4 FTDI</text>
  <text x="385" y="110" text-anchor="middle" font-size="13">USB data / JTAG / UART</text>

  <rect x="560" y="55" width="160" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="640" y="85" text-anchor="middle" font-size="16">K26 target</text>
  <text x="640" y="110" text-anchor="middle" font-size="13">JTAG device</text>

  <rect x="30" y="245" width="170" height="60" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="115" y="280" text-anchor="middle" font-size="15">12 V / 3 A supply</text>

  <rect x="295" y="235" width="180" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="385" y="265" text-anchor="middle" font-size="16">J12</text>
  <text x="385" y="288" text-anchor="middle" font-size="13">main board power</text>

  <path d="M200 140 L295 105" fill="none" stroke="#333" stroke-width="2"/>
  <polygon points="295,105 285,101 288,112" fill="#333"/>
  <text x="245" y="110" text-anchor="middle" font-size="12">USB data</text>

  <path d="M475 95 L560 95" fill="none" stroke="#333" stroke-width="2"/>
  <polygon points="560,95 550,90 550,100" fill="#333"/>
  <text x="518" y="82" text-anchor="middle" font-size="12">JTAG</text>

  <path d="M200 275 L295 275" fill="none" stroke="#333" stroke-width="2"/>
  <polygon points="295,275 285,270 285,280" fill="#333"/>
  <text x="247" y="261" text-anchor="middle" font-size="12">12 V DC</text>

  <path d="M475 270 L640 135" fill="none" stroke="#333" stroke-width="2"/>
  <polygon points="640,135 629,137 636,145" fill="#333"/>
</svg>

**J4 is the data/debug path. J12 is the main-power path. Both paths must be correct before target discovery is meaningful.**

## 4. Connect and power

1. keep the KV260 unpowered;
2. connect the development-host USB data cable to **J4**;
3. connect the 12 V / 3 A supply to **J12**;
4. apply power;
5. observe fan/power-status behavior.

Do not treat one illuminated LED as proof that JTAG works. Power evidence and target-discovery evidence are separate layers.

## 5. Discover the target with the course script

From the repository root:

```bash
vivado -mode batch -nolog -nojournal \
  -source boards/kv260/scripts/detect_target.tcl \
  | tee lab-hw-02-target-detection.txt
```

The script:

1. opens Hardware Manager;
2. connects to the local hw_server;
3. opens an available hardware target;
4. prints devices returned by `get_hw_devices`;
5. requires at least one FPGA device whose Vivado hardware name begins with `xck26`;
6. exits non-zero if no XCK26 FPGA device is found.

You may also see a PS debug object such as `arm_dap_1`. That is useful evidence that the chain is visible, but **it is not the FPGA device and does not satisfy T-HW-002 by itself**.

This lab does **not** require our bitstream to be present. It only proves that the development host can see a real JTAG device.

## 6. Expected Evidence

Retain:

- `lab-hw-02-target-detection.txt`;
- board model + carrier revision;
- Vivado version;
- hardware device name(s), including the `KV260_FPGA_DEVICE=xck26...` line;
- Git commit;
- date.

Passing means an XCK26 FPGA device is present and discovery is repeatable. Power-cycle/reconnect once and rerun to rule out an accidental stale state.

### Save Evidence

Use `boards/kv260/evidence/manifest.example.json` as the T-HW-011 checklist. Copy it to a local evidence filename for LAB-HW-02, fill it from this actual run, and list the retained log/photo/text artifacts. Generated evidence under `boards/kv260/evidence/` is ignored by Git unless deliberately published through a hardware-checkpoint record.

## 7. If it does not work

Debug in this order. **Do not edit RTL first.**

1. **Power** — is J12 actually receiving 12 V from the correct supply?
2. **Cable** — J4, not another USB connector; data-capable cable?
3. **Driver** — did LAB-HW-00 validate the JTAG/cable driver?
4. **Toolchain** — correct Vivado and board data?
5. **Target ownership** — close other Vivado sessions that may own hw_server/JTAG and retry.

`NO_HW_DEVICE` is a bring-up failure, not a neuron-design failure.

## 8. Human Check

1. Why is J12 power still required when USB/J4 is connected?
2. What does “board powered” prove versus “JTAG target discovered”?
3. Why is FlyBrain RTL deliberately absent today?
4. If `get_hw_devices` is empty, which three layers do you check first?

## 9. Official basis

AMD UG1089 Interfaces:  
https://docs.amd.com/r/en-US/ug1089-kv260-starter-kit/Interfaces

AMD UG1089 Powering:  
https://docs.amd.com/r/en-US/ug1089-kv260-starter-kit/Powering-the-Starter-Kit-and-Power-Budgets

AMD DS986 Power and Electrical:  
https://docs.amd.com/r/en-US/ds986-kv260-starter-kit/Power-and-Electrical

AMD DS987 K26 SOM Data Sheet (XCK26 device):  
https://docs.amd.com/r/en-US/ds987-k26-som/Functional-Overview-and-Block-Diagram